# 17 - EASA Validate and Transform

Ingests only approved sources, writes immutable Bronze evidence, applies the signed six-dimensional rule set, quarantines failures, and appends accepted records to Silver. Real-source processing is blocked until the source registry and annual inventory are approved.

In [ ]:
# PARAMETERS - supplied by a governed Data Factory pipeline or an authorized operator.
environment_name = 'dev'
trigger_mode = 'scheduled'
source_id = 'SYNTHETIC-EASA-TEST'
input_path = ''
ingestion_run_id = ''
actor_object_id = ''
config_root = '/lakehouse/default/Files/easa/config'
schema_path = '/lakehouse/default/Files/easa/sql/easa_medallion.sql'
evidence_root = '/lakehouse/default/Files/easa/evidence'

import hashlib
import json
import re
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

assert environment_name in {'dev', 'test', 'prod'}
assert trigger_mode in {'scheduled', 'event'}
assert input_path, 'input_path is required'
assert actor_object_id, 'actor_object_id is required for evidence'
if not ingestion_run_id:
    ingestion_run_id = 'EASA-INGEST-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:12]
validation_run_id = ingestion_run_id.replace('INGEST', 'VALIDATE', 1)

In [ ]:
def read_json(path):
    return json.loads(notebookutils.fs.head(path, 4 * 1024 * 1024))

matrix = read_json(config_root + '/easa_requirements_matrix.json')
source_policy = read_json(config_root + '/easa_approved_sources.json')
deployment = read_json(config_root + '/easa_deployment.json')
environment = deployment['environments'][environment_name]
sources = [item for item in source_policy['sources'] if item['source_id'] == source_id]
assert len(sources) == 1, 'Source must resolve uniquely in the approved source registry'
source = sources[0]
assert source['approved'] is True and source['ingestion_enabled'] is True, 'BLOCKED_SOURCE_NOT_APPROVED'
if source.get('allowed_environments'):
    assert environment_name in source['allowed_environments'], 'BLOCKED_SOURCE_ENVIRONMENT'
if source['domain'] == 'synthetic_test':
    assert environment['allow_synthetic_test_data'] is True and environment_name != 'prod', 'BLOCKED_SYNTHETIC_IN_PROD'
else:
    assert environment['real_source_ingestion_enabled'] is True, 'BLOCKED_REAL_SOURCE_INGESTION_DISABLED'

approved_requirements = [
    item for item in matrix['requirements']
    if item['inventory_approved'] is True
    and item['approval_status'] == 'APPROVED'
    and item['automation_eligibility'] == 'ELIGIBLE'
]
assert approved_requirements, 'BLOCKED_NO_APPROVED_AUTOMATION_INVENTORY'
required_dimensions = {'RECONCILIATION', 'COMPLETENESS', 'VALIDITY', 'DUPLICATE', 'TIMELINESS', 'CROSS_FIELD'}
for requirement in approved_requirements:
    assert 'TODO' not in json.dumps(requirement).upper(), 'BLOCKED_UNRESOLVED_REQUIREMENT_' + requirement['requirement_id']
    dimensions = {rule['rule_type'] for rule in requirement['validation_rules']}
    assert required_dimensions <= dimensions, 'BLOCKED_INCOMPLETE_RULE_SET_' + requirement['requirement_id']

ddl = notebookutils.fs.head(schema_path, 4 * 1024 * 1024)
ddl_without_comments = '\n'.join(line for line in ddl.splitlines() if not line.lstrip().startswith('--'))
for statement in [part.strip() for part in re.split(r';\s*(?:\n|$)', ddl_without_comments) if part.strip()]:
    spark.sql(statement)

In [ ]:
incoming = spark.read.json(input_path)
required_columns = {'source_record_id', 'airport_id', 'requirement_id', 'source_event_at', 'payload'}
missing_columns = required_columns - set(incoming.columns)
assert not missing_columns, 'BLOCKED_INPUT_SCHEMA_MISSING_' + '_'.join(sorted(missing_columns))
allowed_requirement_ids = {item['requirement_id'] for item in approved_requirements}
unknown_requirement_count = incoming.filter(~F.col('requirement_id').isin(sorted(allowed_requirement_ids))).count()
assert unknown_requirement_count == 0, 'BLOCKED_UNAPPROVED_REQUIREMENT_REFERENCE'

bronze = (incoming
    .withColumn('ingestion_run_id', F.lit(ingestion_run_id))
    .withColumn('source_id', F.lit(source_id))
    .withColumn('source_domain', F.lit(source['domain']))
    .withColumn('source_event_at', F.to_timestamp('source_event_at'))
    .withColumn('received_at_utc', F.current_timestamp())
    .withColumn('source_uri', F.lit(input_path))
    .withColumn('payload_json', F.to_json('payload'))
    .withColumn('payload_sha256', F.sha2(F.col('payload_json'), 256))
    .withColumn('source_contract_version', F.lit(source['data_contract_reference']))
    .withColumn('data_classification', F.lit(source['data_classification']))
    .withColumn('is_synthetic', F.lit(source['domain'] == 'synthetic_test'))
    .withColumn('ingested_by', F.lit(actor_object_id))
    .select('ingestion_run_id', 'source_id', 'source_domain', 'airport_id', 'source_record_id', 'requirement_id', 'source_event_at', 'received_at_utc', 'source_uri', 'payload_json', 'payload_sha256', 'source_contract_version', 'data_classification', 'is_synthetic', 'ingested_by'))

existing_keys = spark.table('bronze_easa_source_record').select('source_id', 'source_record_id', 'payload_sha256')
new_bronze = bronze.join(existing_keys, ['source_id', 'source_record_id', 'payload_sha256'], 'left_anti')
new_bronze.drop('requirement_id').write.mode('append').format('delta').saveAsTable('bronze_easa_source_record')

In [ ]:
failure_schema = 'source_record_id string, requirement_id string, airport_id string, quality_dimension string, rule_id string, severity string, failure_code string, payload_sha256 string'
failure_frames = []
quality_results = []
evaluated_at = datetime.now(timezone.utc)

for requirement in approved_requirements:
    requirement_id = requirement['requirement_id']
    subset = new_bronze.filter(F.col('requirement_id') == requirement_id)
    evaluated_count = subset.count()
    rules = requirement['validation_rules']
    for rule in rules:
        dimension = rule['rule_type']
        failed = None
        if dimension == 'DUPLICATE':
            duplicate_ids = subset.groupBy('source_record_id').count().filter(F.col('count') > 1).select('source_record_id')
            failed = subset.join(duplicate_ids, 'source_record_id', 'inner')
        elif dimension == 'COMPLETENESS':
            checks = [F.get_json_object('payload_json', '$.' + field.replace('payload.', '')).isNull() for field in requirement['source_fields']]
            condition = checks[0]
            for check in checks[1:]:
                condition = condition | check
            failed = subset.filter(condition)
        elif dimension == 'RECONCILIATION':
            persisted_count = spark.table('bronze_easa_source_record').filter(F.col('ingestion_run_id') == ingestion_run_id).count()
            failed = subset.limit(1) if persisted_count < new_bronze.count() else subset.limit(0)
        else:
            expression = rule.get('sql_expression')
            assert expression, 'BLOCKED_NON_EXECUTABLE_RULE_' + rule['rule_id']
            assert ';' not in expression and '--' not in expression and '/*' not in expression, 'BLOCKED_UNSAFE_RULE_EXPRESSION_' + rule['rule_id']
            failed = subset.filter('NOT (' + expression + ')')
        failed_count = failed.count()
        quality_results.append((
            hashlib.sha256((validation_run_id + '|' + requirement_id + '|' + rule['rule_id']).encode()).hexdigest(),
            validation_run_id, requirement_id, None, dimension, rule['rule_id'], rule['severity'],
            'PASS' if failed_count == 0 else 'FAIL', evaluated_count, failed_count, evaluated_at,
            hashlib.sha256(json.dumps(rule, sort_keys=True).encode()).hexdigest(), source['domain'] == 'synthetic_test'))
        if failed_count:
            failure_frames.append(failed.select(
                'source_record_id', 'requirement_id', 'airport_id',
                F.lit(dimension).alias('quality_dimension'),
                F.lit(rule['rule_id']).alias('rule_id'),
                F.lit(rule['severity']).alias('severity'),
                F.lit('RULE_FAILED').alias('failure_code'), 'payload_sha256'))

quality_schema = 'quality_result_id string, validation_run_id string, requirement_id string, airport_id string, quality_dimension string, rule_id string, severity string, result_status string, evaluated_record_count long, failed_record_count long, evaluated_at_utc timestamp, rule_version_hash string, is_synthetic boolean'
spark.createDataFrame(quality_results, quality_schema).write.mode('append').format('delta').saveAsTable('gold_easa_quality_result')

failures = spark.createDataFrame([], failure_schema)
for frame in failure_frames:
    failures = failures.unionByName(frame)
if failures.count():
    quarantine = (failures
        .withColumn('quarantine_id', F.sha2(F.concat_ws('|', F.lit(validation_run_id), 'source_record_id', 'rule_id'), 256))
        .withColumn('validation_run_id', F.lit(validation_run_id))
        .withColumn('ingestion_run_id', F.lit(ingestion_run_id))
        .withColumn('source_id', F.lit(source_id))
        .withColumn('failure_detail', F.concat(F.lit('Signed rule failed: '), F.col('rule_id')))
        .withColumn('quarantined_at_utc', F.current_timestamp())
        .withColumn('resolved_at_utc', F.lit(None).cast('timestamp'))
        .withColumn('resolution_evidence_reference', F.lit(None).cast('string'))
        .withColumn('is_synthetic', F.lit(source['domain'] == 'synthetic_test'))
        .select('quarantine_id', 'validation_run_id', 'ingestion_run_id', 'requirement_id', 'source_id', 'source_record_id', 'airport_id', 'quality_dimension', 'rule_id', 'failure_code', 'failure_detail', 'payload_sha256', 'quarantined_at_utc', 'resolved_at_utc', 'resolution_evidence_reference', 'is_synthetic'))
    quarantine.write.mode('append').format('delta').saveAsTable('silver_easa_quarantine')

blocking_ids = failures.filter(F.col('severity') == 'BLOCKING').select('source_record_id').distinct()
accepted = new_bronze.join(blocking_ids, 'source_record_id', 'left_anti')
silver = (accepted
    .withColumn('validation_run_id', F.lit(validation_run_id))
    .withColumn('conformed_payload_json', F.col('payload_json'))
    .withColumn('validation_rule_set_version', F.sha2(F.lit(json.dumps(matrix, sort_keys=True)), 256))
    .withColumn('validated_at_utc', F.current_timestamp())
    .select('validation_run_id', 'ingestion_run_id', 'requirement_id', 'source_id', 'source_domain', 'airport_id', 'source_record_id', 'source_event_at', 'received_at_utc', 'conformed_payload_json', 'payload_sha256', 'validation_rule_set_version', 'validated_at_utc', 'is_synthetic'))
silver.write.mode('append').format('delta').saveAsTable('silver_easa_validated_record')

In [ ]:
manifest = {
    'ingestion_run_id': ingestion_run_id,
    'validation_run_id': validation_run_id,
    'environment': environment_name,
    'trigger_mode': trigger_mode,
    'source_id': source_id,
    'input_count': incoming.count(),
    'bronze_appended_count': new_bronze.count(),
    'silver_accepted_count': silver.count(),
    'quarantined_failure_count': failures.count(),
    'matrix_sha256': hashlib.sha256(json.dumps(matrix, sort_keys=True).encode()).hexdigest(),
    'source_policy_sha256': hashlib.sha256(json.dumps(source_policy, sort_keys=True).encode()).hexdigest(),
    'observed_at_utc': datetime.now(timezone.utc).isoformat(),
}
manifest_text = json.dumps(manifest, sort_keys=True, indent=2)
manifest['evidence_sha256'] = hashlib.sha256(manifest_text.encode()).hexdigest()
notebookutils.fs.put(evidence_root + '/' + ingestion_run_id + '.json', json.dumps(manifest, indent=2), True)
print(json.dumps(manifest, indent=2))